![image.png](https://i.imgur.com/a3uAqnb.png)

# **🏭 Exercise: Steel Defect Classification with Pretrained Model (EfficientNet)**

> **Your turn!** Yesterday you trained a CNN from scratch on the NEU steel defect dataset. Today let's see what **transfer learning** can do on the *same* dataset.

In this exercise, we will:

✅ Build a **Dataset** for the **NEU Steel Defects dataset** (6 defect classes)  
✅ Use a **pretrained EfficientNet-B0 model** for classification  
✅ Train and evaluate the model using **three different fine-tuning strategies**  
✅ Compare the results to the simple CNN you built yesterday — was the effort worth it?


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import kagglehub
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# Download the dataset
path = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")
path


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.9MB/s]

Extracting files...


'/root/.cache/kagglehub/datasets/kaustubhdikshit/neu-surface-defect-database/versions/1'

In [ ]:
# The downloaded folder contains a top-level "NEU-DET" folder
DATA_DIR = os.path.join(path, "NEU-DET")
print("Contents of DATA_DIR:", os.listdir(DATA_DIR))
print("Train classes:", os.listdir(os.path.join(DATA_DIR, 'train', 'images')))


Contents of DATA_DIR: ['validation', 'train']
Train classes: ['rolled-in_scale', 'scratches', 'pitted_surface', 'crazing', 'patches', 'inclusion']


## 1️⃣ Dataset Class

Unlike the Cats & Dogs dataset (which used a CSV), the NEU dataset is organized as **folders of images** — one folder per class. PyTorch's `ImageFolder` is built exactly for this layout, so we'll use it instead of writing our own `Dataset` class. ✨

```
NEU-DET/
├── train/images/
│   ├── crazing/      → image1.jpg, image2.jpg, ...
│   ├── inclusion/
│   ├── patches/
│   ├── pitted_surface/
│   ├── rolled-in_scale/
│   └── scratches/
└── validation/images/
    └── ... (same 6 folders)
```


#### In the transformation pipeline, we use **specific mean and standard deviation values**

## **🔹 Where Do These Values Come From?**
✅ These values were found using the **ImageNet dataset**, a large-scale dataset used for training **pretrained models** like ResNet, EfficientNet, and VGG.  
✅ **Reference**: [ImageNet Dataset](https://paperswithcode.com/dataset/imagenet)  

## **🔹 Why Do We Use Them?**
1️⃣ **Pretrained Models Expect These Values**  
   - If a model was trained using a certain mean & std, we should use the **same values** for inference/training.
   - This ensures the input distribution matches what the model was trained on.
   - We are finetuning an EfficientNet here, which was pretrained on Imagenet, so will use these values.

🤔 **Think about it:** ImageNet contains photos of cats, dogs, planes, cars... NOT steel surfaces. Will a model pretrained on it still help us? You'll find out below 👀


### Let's display some images

## 2️⃣ Model Class

#### Instead of training from scratch, we use **EfficientNet-B0** with a modified output layer.

Compared to the cats & dogs lab, the only change is the final layer — we now output **???? classes** instead of 1 (binary).


## 3️⃣ Training and Validation Loops

These loops are slightly different from yesterday's cats & dogs lab because we have **6 classes instead of 2**:

| | Cats & Dogs (binary) | NEU Defects (multi-class) |
|---|---|---|
| Loss | `BCEWithLogitsLoss` | ? |
| Output shape | `[batch, 1]` → squeeze | `[batch, 6]` → no squeeze |
| Predictions | `sigmoid(out) > 0.5` | `argmax(out)` |
| Labels dtype | `float` | `long` (default) |


## 4️⃣ Running Training — Strategy 1: Train Everything

We start by training **all the parameters** — the pretrained weights are just a starting point. This usually gives the best accuracy but is slow and needs more data.


### Plot loss and some predictions

## Strategy 2: Freeze Everything, Train Only Classifier Head

This is the **fastest** approach for transfer learning. The pretrained backbone acts as a fixed feature extractor, and you only train the final classification layer. Perfect for small datasets (few hundred images) or when you want quick results. Low risk of overfitting since you're only training ~1,000 parameters.

**When to use:** Small datasets, limited compute, quick experiments, or when your task is similar to ImageNet.

🤔 **Prediction time:** Steel defects don't really look like ImageNet photos... will this still work? Make a guess before running it. 👀


## Strategy 3: Fine-Tune Last Layers + Classifier

This approach unfreezes the **last few blocks** of the backbone along with the classifier. The early layers (which learn generic features like edges) stay frozen, while later layers adapt to your specific task. This gives better accuracy than Strategy 1 but requires more data and longer training.

**When to use:** Medium datasets (1000+ images), when your domain differs from ImageNet, or when just training the classifier head doesn't give enough accuracy.

🏭 **Steel defects ≠ cats & dogs.** ImageNet's edge/texture detectors are still useful (early layers), but the higher-level "is this a face?" features are useless to us — so we let those late layers re-learn for our domain.


## See Some Predictions

### Contributed by: Yazan Alshuaibi